# 01 – LangChain Fundamentals

**Learning track:** Generative AI → Agentic AI → LangChain → LangGraph  
**Environment:** Google Colab  
**LLM provider:** Google Gemini  
**Status:** Learning in progress

## Learning objectives

By the end of this notebook, you should be able to:

1. Set up LangChain in Google Colab.
2. Connect a Gemini chat model.
3. Create an agent with `create_agent()`.
4. Define custom tools with `@tool`.
5. Request structured responses with Pydantic.
6. Extend an agent's state with application-specific fields.

> Learning approach: run each section, inspect the output, and write down what you learned. The tools in this notebook are deliberately simple demonstrations; they are not production integrations.

## 1. Environment setup

Install LangChain and the Google GenAI integration.

Run this cell once per fresh Colab runtime. If you install or upgrade packages after imports have already run, restart the runtime before continuing.

In [ ]:
%pip install -qU langchain langchain-google-genai pydantic

## 2. Configure the Gemini API key

For safer key handling, use **Colab Secrets**:

1. Open the key icon in Colab's left sidebar.
2. Create a secret named `GEMINI_API_KEY`.
3. Add your Gemini API key as its value.
4. Enable notebook access for that secret.

Do not paste API keys directly into notebook cells or commit them to GitHub.

In [ ]:
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")

print("Gemini API key loaded:", bool(os.environ.get("GOOGLE_API_KEY")))

## 3. Create your first LangChain agent

An **agent** uses a language model to decide how to respond and, when appropriate, whether to call an available tool.

In this first example, the agent has no tools. We are checking that the Gemini model connection works.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    tools=[],
    system_prompt="You are a helpful assistant. Explain concepts clearly and concisely."
)

result = agent.invoke({
    "messages": [
        {"role": "user", "content": "What is LangChain? Explain in three bullet points."}
    ]
})

print(result["messages"][-1].content)

## 4. Create a custom tool with `@tool`

A tool is a Python function that an agent can choose to call. The function's docstring helps describe the tool to the model.

The function below is a **mock weather tool**. It does not call a weather service or retrieve live conditions.

In [ ]:
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """Return a demonstration weather message for the given city."""
    return f"Demo result: the weather in {city} is sunny."

weather_agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    tools=[get_weather],
    system_prompt="You are a helpful assistant. Use the weather tool when the user asks about weather."
)

weather_result = weather_agent.invoke({
    "messages": [
        {"role": "user", "content": "What's the weather in San Francisco?"}
    ]
})

print(weather_result["messages"][-1].content)

### Inspect the tool call

The following cell displays the message types and any tool-call metadata returned during the agent run. A tool call is evidence that the model requested a tool; the tool result is recorded in the message history.

In [ ]:
for message in weather_result["messages"]:
    print("Message type:", message.type)
    print("Content:", message.content)
    print("Tool calls:", getattr(message, "tool_calls", []))
    print("-" * 60)

## 5. Return structured output with Pydantic

Sometimes an application needs predictable fields rather than a free-form paragraph. LangChain supports structured responses using a Pydantic model.

Here, `confidence` is the model's self-reported estimate. It is **not** a statistically calibrated probability. The `source` field should not be treated as verified evidence unless the agent actually consults a reliable source.

In [ ]:
from pydantic import BaseModel, Field

class Answer(BaseModel):
    answer: str = Field(description="A concise answer to the user's question")
    confidence: float = Field(description="The model's self-assessed confidence from 0 to 1")
    source: str = Field(description="The source or basis used for the answer")

structured_agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    tools=[],
    system_prompt=(
        "You are a helpful assistant. Be concise and accurate. "
        "If you do not have an external source, say that the answer is based on your model knowledge."
    ),
    response_format=Answer
)

structured_result = structured_agent.invoke({
    "messages": [
        {"role": "user", "content": "What is a model in LangChain?"}
    ]
})

response = structured_result["structured_response"]

print(f"Answer: {response.answer}")
print(f"Confidence: {response.confidence}")
print(f"Source: {response.source}")

## 6. Extend the agent state

`AgentState` defines the state fields used by an agent. You can extend it with application-specific information.

The built-in `messages` field is inherited from `AgentState`. The additional fields below are optional, so they do not have to be supplied in every invocation.

> Custom state is not automatically durable memory. To preserve state across invocations, configure an appropriate checkpointer and use a thread ID.

In [ ]:
from typing import NotRequired
from langchain.agents import AgentState, create_agent

class MyState(AgentState):
    user_id: NotRequired[str]
    counter: NotRequired[int]

state_agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    tools=[],
    system_prompt="You are a helpful assistant. Answer clearly and concisely.",
    state_schema=MyState
)

state_result = state_agent.invoke({
    "messages": [
        {"role": "user", "content": "What is an agent state in LangChain?"}
    ],
    "user_id": "learner_001",
    "counter": 1
})

print("User ID:", state_result.get("user_id"))
print("Counter:", state_result.get("counter"))
print("Assistant:", state_result["messages"][-1].content)

## 7. Practice exercises

Try these exercises yourself before looking for a solution:

1. Change the system prompt and compare the agent's response.
2. Add a second mock tool, such as `get_time(city: str)`.
3. Ask a question that should cause the agent to call each tool.
4. Add a `category` field to the `Answer` Pydantic model.
5. Add an optional `task_status` field to `MyState`.
6. Inspect the messages returned by each `invoke()` call.

## 8. Key takeaways

- `create_agent()` creates a tool-capable agent.
- `@tool` exposes a Python function as a tool.
- `response_format` can return a validated structured response.
- Extending `AgentState` adds custom fields to agent state.
- A mock tool demonstrates tool calling, but does not connect to a real external service.
- Custom state and persistent memory are related concepts, but they are not the same.

## 9. Learning notes

Use this section to record your own understanding, errors, and questions.

- **What I understood:**
- **What I found difficult:**
- **What I want to explore next:**